In [ ]:
import os
import csv
import time
import requests

PODSCAN_API_BASE = "https://podscan.fm/api/v1"
PODSCAN_API_KEY = os.getenv("PODSCAN_API_KEY")
if not PODSCAN_API_KEY:
    raise EnvironmentError("Set the PODSCAN_API_KEY environment variable before running this notebook.")

headers = {"Authorization": f"Bearer {PODSCAN_API_KEY}"}

params = {
    "language": "en",
    "min_last_episode_posted_at": "2026-01-01",
    "min_audience_size": 5000,
    "per_page": 500,
    "order_by": "audience_size",
    "order_dir": "desc",
}

all_podcasts = []
page = 1

while True:
    params["page"] = page
    resp = requests.get(f"{PODSCAN_API_BASE}/podcasts/search", headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    podcasts = data["podcasts"]
    all_podcasts.extend(podcasts)

    total = data["pagination"]["total"]
    last_page = data["pagination"]["last_page"]
    print(f"Page {page}/{last_page} — fetched {len(podcasts)} podcasts (total so far: {len(all_podcasts)}/{total})")

    if page >= last_page:
        break
    page += 1
    time.sleep(0.5)  # be polite to the API

print(f"\nDone! Fetched {len(all_podcasts)} podcasts total.")

# Write to CSV
output_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, "podscan_podcasts.csv")

fieldnames = [
    "podcast_name", "publisher_name", "episode_count", "audience_size",
    "last_posted_at", "language", "podcast_has_guests", "podcast_url",
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for pod in all_podcasts:
        writer.writerow({
            "podcast_name": pod["podcast_name"],
            "publisher_name": pod["publisher_name"],
            "episode_count": pod["episode_count"],
            "audience_size": pod["reach"].get("audience_size", ""),
            "last_posted_at": pod["last_posted_at"],
            "language": pod["language"],
            "podcast_has_guests": pod["podcast_has_guests"],
            "podcast_url": pod["podcast_url"],
        })

print(f"CSV saved to: {csv_path}")